# ¿Sirve φ⁴ como medidor de riesgo? Validación como alerta temprana

**Pregunta:** ¿los indicadores del φ⁴ calculados hoy anticipan la turbulencia de las próximas 4 semanas mejor que
lo que ya usa la industria?

**Universo:** las 20 mayores acciones del S&P 500 (las mismas de `sp500_top20_riesgo.ipynb`), 10 años. La
turbulencia se mide en un portafolio equiponderado de las 20.

| Tipo | Indicador (ventana de 120 días hasta la fecha t) |
| --- | --- |
| **φ⁴** | acoplamiento medio w/√(μμ) · fuerza del mayor hub · dispersión de log μ_i · curtosis transversal del modelo |
| Referencia de la literatura | correlación media · *absorption ratio* (Kritzman et al., 2011; varianza absorbida por los 4 primeros vectores propios) |
| **Vara mínima** | volatilidad actual del portafolio (últimos 20 días) |

**Objetivos:** volatilidad realizada y caída máxima en t+1…t+H, con **H = 20 y 60 días**, para dos activos: el
portafolio equiponderado de las 20 y el **SPY**. "Evento de estrés" = el 10% de las fechas con mayor caída siguiente.

**Versión favorable a los indicadores (literatura):** además del nivel se prueba el **cambio estandarizado**
(Kritzman et al., 2011: media de 15 días menos media de 1 año, dividido por la desviación de 1 año) del absorption
ratio, de la correlación media y del acoplamiento φ⁴. En los trabajos originales el absorption ratio anticipa sobre
todo a través de estos saltos y a horizontes más largos.

**Por qué la volatilidad actual es la vara mínima:** la volatilidad se agrupa en el tiempo, así que la de hoy ya
anticipa buena parte de la de mañana. Un indicador solo sirve si agrega información **además** de ella.

**Cómo se decide:**
1. **Dentro de muestra:** regresión objetivo ~ volatilidad actual + indicador, con errores de Newey–West (los
   objetivos se solapan).
2. **Fuera de muestra (la prueba que importa):** regresión con ventana expansiva, sin que el objetivo de
   entrenamiento se solape con la fecha evaluada. R²_oos > 0 significa que el indicador mejora al modelo base;
   la prueba de Clark–West dice si la mejora es significativa.
3. Dos modelos base: (a) solo volatilidad actual; (b) volatilidad + correlación media + absorption ratio + su
   cambio estandarizado. Pasar (b) es lo que haría del φ⁴ un medidor de riesgo con valor propio.

**Tiempo estimado:** 3–15 minutos (el φ⁴ se reajusta cada `STEP` días). Usa la misma caché de precios que el
notebook del mapa de riesgo.

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import phi4finance
from phi4finance import RollingPhi4, load_returns
from phi4finance.earlywarning import (absorption_ratio, auc, average_correlation, forward_max_drawdown,
                                      forward_realized_vol, hac_ols, oos_r2, rolling_indicator,
                                      trailing_realized_vol)

TICKERS = ["NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "AVGO", "TSLA", "MU", "BRK-B",
           "LLY", "AMD", "JPM", "WMT", "V", "INTC", "XOM", "JNJ", "MA", "ABBV"]
MARKET = "SPY"
TODAY = pd.Timestamp.today().normalize()
START = (TODAY - pd.DateOffset(years=11)).strftime("%Y-%m-%d")
END = (TODAY + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
DATA_DIR, RESULTS_DIR = Path("data"), Path("results")

FULL = False
WIN = 120                       # ventana de los indicadores (días)
STEP = 5 if FULL else 10        # cada cuántos días se calcula (φ⁴ se reajusta)
HORIZONS = [20, 60]             # horizontes del objetivo (días hábiles)
K_AR = 4                        # vectores propios del absorption ratio (20 acciones / 5)
EVENT_Q = 0.90                  # evento = caída siguiente por encima de su percentil 90
MIN_TRAIN_YEARS = 3             # historia mínima antes de la primera predicción fuera de muestra
print("phi4finance", phi4finance.__version__, "| FULL =", FULL)

In [ ]:
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
C_PHI4, C_BENCH, C_BASE, C_TARGET = "#eb6834", "#2a78d6", "#898781", "#0b0b0b"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 1.4, "legend.frameon": False, "legend.labelcolor": INK2,
    "font.size": 10, "figure.dpi": 110,
})


def styled(df, fmt):
    try:
        return df.style.format(fmt)
    except (AttributeError, ImportError):
        return df.round(4)

## 1. Datos y portafolio

Retornos simples diarios de cierres ajustados; el portafolio se rebalancea cada día a 5% por acción.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", UserWarning)
    RALL = load_returns(TICKERS + [MARKET], start=START, end=END, log=False, cache_dir=DATA_DIR)
for x in w:
    if issubclass(x.category, UserWarning) and "unclosed" not in str(x.message):
        print("aviso:", x.message)
TICKERS = [t for t in TICKERS if t in RALL.columns]
R = RALL[TICKERS]
MKT = RALL[MARKET]
P = R.mean(axis=1)                     # 11 años: el primero sirve de arranque para ventanas y cambios de 1 año
print(f"{len(R)} días, {R.index[0].date()} a {R.index[-1].date()}, {R.shape[1]} acciones")

## 2. Indicadores

El φ⁴ se ajusta con `RollingPhi4` sobre los retornos de cada ventana (sin filtro de volatilidad: queremos medir el
estado del mercado tal cual), partiendo de los parámetros de la fecha anterior. De cada ajuste salen:

- **acoplamiento medio**: media de w_ij/√(μ_i μ_j) (correlación parcial media que implica el modelo);
- **fuerza del mayor hub**: la acción más conectada, Σ_j |w_ij|/√(μ_i μ_j);
- **dispersión de log μ_i**: cuán distintas son las volatilidades entre acciones (es lo que genera curtosis
  transversal en el modelo);
- **curtosis del modelo**: curtosis transversal media de 1000 configuraciones muestreadas (el indicador de la Fig. 1
  del paper).

In [ ]:
t0 = time.time()
roll = RollingPhi4(window=WIN, step=STEP, n_samples=1000, burn=200, l2=0.01, verbose=True,
                   model_kw={"mu_global": False, "lam_global": False}, fit_kw={"n_grid": 101})
roll.fit(R)
res = roll.results_
dates = res.index
print(f"{len(res)} ajustes φ⁴ en {time.time() - t0:.0f} s")

IND = pd.DataFrame({
    "φ⁴ acoplamiento medio": res.coupling_mean,
    "φ⁴ mayor hub": res.hub_strength_max,
    "φ⁴ dispersión log μ": res.log_mu_std,
    "φ⁴ curtosis modelo": res.model_market_kurtosis,
    "curtosis datos": res.data_market_kurtosis,
    "correlación media": rolling_indicator(R, average_correlation, WIN, dates),
    "absorption ratio": rolling_indicator(R, lambda x: absorption_ratio(x, K_AR), WIN, dates),
    "vol. actual (20d)": trailing_realized_vol(P, 20).reindex(dates),
})
# Cambio estandarizado (Kritzman et al., 2011): (media 15 días − media 1 año) / desviación 1 año.
# El absorption ratio y la correlación media se calculan a diario para esto; el φ⁴ en su grilla de STEP días.
daily = R.index[WIN - 1:]
AR_daily = rolling_indicator(R, lambda x: absorption_ratio(x, K_AR), WIN, daily)
CM_daily = rolling_indicator(R, average_correlation, WIN, daily)
shift_daily = lambda x: (x.rolling(15).mean() - x.rolling(252).mean()) / x.rolling(252).std()
n15, n1y = max(1, round(15 / roll.step)), round(252 / roll.step)
shift_grid = lambda x: (x.rolling(n15).mean() - x.rolling(n1y).mean()) / x.rolling(n1y).std()
IND["Δ absorption ratio"] = shift_daily(AR_daily).reindex(dates)
IND["Δ correlación media"] = shift_daily(CM_daily).reindex(dates)
IND["Δ φ⁴ acoplamiento"] = shift_grid(IND["φ⁴ acoplamiento medio"])

PHI4 = [c for c in IND.columns if "φ⁴" in c]
BENCH = ["correlación media", "absorption ratio", "Δ absorption ratio", "Δ correlación media", "curtosis datos"]
print("\nCorrelación entre indicadores (¿el φ⁴ mide algo distinto?):")
IND.corr(method="spearman").round(2)

## 3. Objetivos y evidencia dentro de muestra

Para cada activo (portafolio EW y SPY), horizonte H (20 y 60 días) y fecha t: volatilidad realizada (log) y caída
máxima en t+1…t+H. El gráfico muestra, para el portafolio a 60 días, cada indicador y la volatilidad siguiente,
ambos estandarizados en el mismo eje.

In [ ]:
ASSETS = {"portafolio EW": P, "SPY": MKT.loc[R.index]}
DATA = {}
for a, ser in ASSETS.items():
    vol_now = np.log(trailing_realized_vol(ser, 20)).reindex(dates)
    for h in HORIZONS:
        Y = pd.DataFrame({"log vol. siguiente": np.log(forward_realized_vol(ser, h)),
                          "caída máx. siguiente": forward_max_drawdown(ser, h)}).reindex(dates)
        Dk = IND.join(Y).assign(**{"log vol. actual": vol_now}).dropna()
        Dk = Dk.loc[Dk.index > Dk.index[-1] - pd.DateOffset(years=10)]
        Dk["evento"] = Dk["caída máx. siguiente"] > Dk["caída máx. siguiente"].quantile(EVENT_Q)
        DATA[(a, h)] = Dk
for k, Dk in DATA.items():
    print(f"{k[0]:14s} H={k[1]:2d}: {len(Dk)} fechas ({Dk.index[0].date()} a {Dk.index[-1].date()}), {Dk.evento.sum()} eventos")

zs = lambda x: (x - x.mean()) / x.std()
D = DATA[("portafolio EW", 60)]
show = ["φ⁴ acoplamiento medio", "Δ φ⁴ acoplamiento", "φ⁴ curtosis modelo", "correlación media", "Δ absorption ratio"]
fig, axes = plt.subplots(len(show), 1, figsize=(11, 1.9 * len(show)), sharex=True)
for ax, c in zip(axes, show):
    ax.plot(D.index, zs(D["log vol. siguiente"]), color=C_TARGET, lw=0.9, alpha=0.55, label="log vol. siguiente (60d)")
    ax.plot(D.index, zs(D[c]), color=C_PHI4 if c in PHI4 else C_BENCH, lw=1.4, label=c)
    for d in D.index[D.evento]:
        ax.axvspan(d, d + pd.Timedelta(days=roll.step * 1.4), color="#e34948", alpha=0.08, lw=0)
    ax.set_title(f"{c}  (ρ Spearman con vol. siguiente = {spearmanr(D[c], D['log vol. siguiente'])[0]:.2f})", fontsize=10)
    ax.legend(loc="upper left", fontsize=7, ncol=2)
axes[-1].set_xlabel("Fecha (bandas rojas: fechas previas a un evento de estrés del portafolio a 60 días)")
fig.tight_layout(); plt.show()

In [ ]:
rows = []
for (a, h), Dk in DATA.items():
    lags = int(np.ceil(h / roll.step))
    for target in ["log vol. siguiente", "caída máx. siguiente"]:
        for c in PHI4 + BENCH:
            r = hac_ols(Dk[target], pd.DataFrame({"log vol. actual": Dk["log vol. actual"], c: zs(Dk[c])}), lags=lags)
            rows.append({"activo": a, "H": h, "objetivo": target, "indicador": c,
                         "coef. (por 1σ)": r.loc[c, "coef"], "t (Newey–West)": r.loc[c, "t"], "p": r.loc[c, "p"]})
tab_in = pd.DataFrame(rows).set_index(["activo", "H", "objetivo", "indicador"])
sig = tab_in[tab_in.p < 0.05]
print(f"Dentro de muestra, controlando por la volatilidad actual: {len(sig)} de {len(tab_in)} combinaciones con p < 0.05")
styled(sig, {"coef. (por 1σ)": "{:.4f}", "t (Newey–West)": "{:.2f}", "p": "{:.3f}"})

## 4. Fuera de muestra (la prueba que decide)

En cada fecha se reestima la regresión con la historia disponible (al menos 3 años), dejando fuera las observaciones
cuyo objetivo se solapa con la fecha evaluada, y se predice el objetivo. Se compara el error con el de un modelo base:

- **Base A**: solo la volatilidad actual del activo.
- **Base B**: volatilidad actual + correlación media + absorption ratio + Δ absorption ratio.

R²_oos > 0 y p (Clark–West) < 0.05 = el indicador aporta información nueva y utilizable. Los mapas resumen las 8
combinaciones (activo × horizonte × objetivo).

In [ ]:
baseA = ["log vol. actual"]
baseB = ["log vol. actual", "correlación media", "absorption ratio", "Δ absorption ratio"]
rows = []
for (a, h), Dk in DATA.items():
    min_train = int((Dk.index < Dk.index[0] + pd.DateOffset(years=MIN_TRAIN_YEARS)).sum())
    gap = int(np.ceil(h / roll.step))
    for target in ["log vol. siguiente", "caída máx. siguiente"]:
        for c in PHI4 + BENCH:
            for base_name, base in [("A", baseA), ("B", baseB)]:
                if c in base:
                    continue
                r = oos_r2(Dk[target], Dk[base], Dk[base + [c]], min_train, gap)
                rows.append({"activo": a, "H": h, "objetivo": target, "base": base_name, "indicador": c,
                             "R² oos": r["r2_oos"], "p (Clark–West)": r["p_cw"], "n": r["n"]})
tab_oos = pd.DataFrame(rows).set_index(["activo", "H", "objetivo", "base", "indicador"])

def grid(base, inds):
    t = tab_oos.xs(base, level="base").reset_index()
    t["combo"] = t.activo.str.replace("portafolio ", "") + " · " + t.H.astype(str) + "d · " + \
                 t.objetivo.str.replace(" siguiente", "")
    return (t.pivot(index="indicador", columns="combo", values="R² oos").reindex(inds),
            t.pivot(index="indicador", columns="combo", values="p (Clark–West)").reindex(inds))

DIV = plt.matplotlib.colors.LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
fig, axes = plt.subplots(2, 1, figsize=(12, 9.5), gridspec_kw={"height_ratios": [len(PHI4 + BENCH), len(PHI4)]})
for ax, base, inds, title in [(axes[0], "A", PHI4 + BENCH, "Base A (vol. actual): R² fuera de muestra"),
                              (axes[1], "B", PHI4 + ["curtosis datos", "Δ correlación media"],
                               "Base B (vol. + corr. + AR + ΔAR): R² fuera de muestra")]:
    G, Pv = grid(base, inds)
    lim = max(0.05, np.nanmax(np.abs(G.values)))
    im = ax.imshow(G.values, cmap=DIV, vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(G.shape[1]), G.columns, rotation=25, ha="right", fontsize=8)
    ax.set_yticks(range(G.shape[0]), [f"{i}" for i in G.index], fontsize=8)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            v, p = G.values[i, j], Pv.values[i, j]
            if np.isfinite(v):
                ok = v > 0 and p < 0.05            # mejora y es significativa
                ax.text(j, i, f"{v:+.1%}{'*' if ok else ''}", ha="center", va="center", fontsize=7,
                        color=INK, fontweight="bold" if ok else "normal")
    ax.grid(False); ax.set_title(title + "  (azul = mejora; * = mejora con p < 0.05)")
    fig.colorbar(im, ax=ax, fraction=0.02, format=plt.matplotlib.ticker.PercentFormatter(1.0))
fig.tight_layout(); plt.show()

aucs = pd.DataFrame({f"{a.replace('portafolio ', '')} · {h}d": pd.Series({c: auc(Dk[c], Dk.evento) for c in PHI4 + BENCH + ["log vol. actual"]})
                     for (a, h), Dk in DATA.items()})
print("AUC para anticipar eventos de estrés (0.5 = azar):")
styled(aucs, "{:.2f}")

## 5. Veredicto

La celda siguiente escribe el veredicto a partir de los números de tu corrida.

In [ ]:
passA = tab_oos.xs("A", level="base").query("`R² oos` > 0 and `p (Clark–West)` < 0.05")
passB = tab_oos.xs("B", level="base").query("`R² oos` > 0 and `p (Clark–West)` < 0.05")
n_combos = len(DATA) * 2
print(f"Combinaciones evaluadas por indicador: {n_combos} (2 activos × {len(HORIZONS)} horizontes × 2 objetivos)\n")
for c in PHI4 + BENCH:
    a = passA.xs(c, level="indicador") if c in passA.index.get_level_values("indicador") else passA.iloc[:0]
    b = passB.xs(c, level="indicador") if c in passB.index.get_level_values("indicador") else passB.iloc[:0]
    where = "; ".join(f"{i[0]} {i[1]}d {i[2]}" for i in b.index) if len(b) else ""
    tag = "φ⁴" if c in PHI4 else "ref"
    extra = ""
    if c in PHI4 or c in ("curtosis datos", "Δ correlación media"):
        extra = f" | aporta sobre corr.+AR+ΔAR en {len(b)}/{n_combos}" + (f": {where}" if where else "")
    print(f"[{tag}] {c:24s} mejora a la vol. actual en {len(a)}/{n_combos}{extra}")
print("\nCriterio: un indicador φ⁴ tiene valor propio como alerta temprana si aporta sobre la base B en varias "
      "combinaciones y no en una sola (con 8 pruebas por indicador, un acierto aislado puede ser azar).")

RESULTS_DIR.mkdir(exist_ok=True)
out = RESULTS_DIR / f"validacion_riesgo_sistemico_{R.index[-1].date()}.xlsx"
try:
    with pd.ExcelWriter(out) as xw:
        tab_oos.to_excel(xw, sheet_name="fuera_de_muestra")
        tab_in.to_excel(xw, sheet_name="dentro_de_muestra")
        aucs.to_excel(xw, sheet_name="auc")
        IND.to_excel(xw, sheet_name="indicadores")
    print("\nguardado en", out)
except ImportError:
    tab_oos.to_csv(out.with_suffix(".csv")); print("\nguardado en", out.with_suffix(".csv"))